In [2]:
import pandas as pd
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [3]:
train_data = pd.read_csv(r"C:\Users\pc gamer\Downloads\scotlab\train-test.csv")
train_data.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [4]:
val_data = pd.read_csv(r"C:\Users\pc gamer\Downloads\scotlab\validation.csv")
val_data.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541


In [5]:
print('train_data:', train_data['date'].min(), 'to', train_data['date'].max())
print('val_data:', val_data['date'].min(), 'to', val_data['date'].max())

train_data: 2025-01-01 to 2025-10-31
val_data: 2025-11-01 to 2025-12-31


In [6]:
val_data.isnull().sum()


load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64

In [7]:
train_data.isnull().sum()

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

In [8]:
train_data[train_data['weight'].isna()]['date'].value_counts().head(20)

date
2025-05-07    5
2025-09-08    5
2025-01-07    4
2025-02-16    4
2025-02-01    4
2025-02-23    4
2025-10-03    4
2025-01-13    4
2025-01-08    4
2025-07-09    4
2025-03-12    3
2025-06-25    3
2025-06-23    3
2025-03-22    3
2025-02-05    3
2025-01-12    3
2025-01-05    3
2025-10-14    3
2025-06-28    3
2025-08-28    3
Name: count, dtype: int64

In [9]:
train_data.groupby('equipment')['weight'].median()

equipment
Dry Van    31373.0
Flatbed    31462.5
Reefer     31524.0
Name: weight, dtype: float64

In [10]:
train_data.groupby('date')['market_index'].nunique().head(10)

date
2025-01-01    144
2025-01-02    164
2025-01-03    154
2025-01-04    152
2025-01-05    151
2025-01-06    166
2025-01-07    159
2025-01-08    186
2025-01-09    146
2025-01-10    171
Name: market_index, dtype: int64

In [11]:
train_data.groupby('date')['market_index'].mean().head(20)

date
2025-01-01    0.961224
2025-01-02    0.986606
2025-01-03    0.940339
2025-01-04    0.857702
2025-01-05    0.801855
2025-01-06    0.813340
2025-01-07    0.888123
2025-01-08    0.973030
2025-01-09    1.000927
2025-01-10    0.956976
2025-01-11    0.871767
2025-01-12    0.809704
2025-01-13    0.827929
2025-01-14    0.906950
2025-01-15    0.989950
2025-01-16    1.018295
2025-01-17    0.974304
2025-01-18    0.887977
2025-01-19    0.829469
2025-01-20    0.844291
Name: market_index, dtype: float64

In [12]:
train_data['date'] = pd.to_datetime(train_data['date'])
train_data['dayofweek'] = train_data['date'].dt.dayofweek
print(train_data.groupby('dayofweek')['market_index'].mean())

dayofweek
0    0.999586
1    1.075435
2    1.153226
3    1.178978
4    1.134310
5    1.047481
6    0.988633
Name: market_index, dtype: float64


In [13]:
train_data.groupby('dayofweek')['posted_rate'].mean().head(20)

dayofweek
0    2349.308676
1    2372.890626
2    2405.086706
3    2403.265599
4    2376.849855
5    2373.714334
6    2335.073176
Name: posted_rate, dtype: float64

In [14]:
print(train_data[['distance', 'weight', 'market_index', 'quote_signal', 'posted_rate']].corr()['posted_rate'])

distance        0.908519
weight          0.034840
market_index    0.034165
quote_signal   -0.039858
posted_rate     1.000000
Name: posted_rate, dtype: float64


In [15]:
train_data['rate_per_mile'] = train_data['posted_rate'] / train_data['distance']
print(train_data[['rate_per_mile','weight', 'market_index', 'quote_signal', 'posted_rate']].corr()['rate_per_mile'])

rate_per_mile    1.000000
weight           0.074517
market_index     0.083478
quote_signal     0.049342
posted_rate      0.003147
Name: rate_per_mile, dtype: float64


In [16]:
print(train_data.groupby(pd.cut(train_data['distance'],bins=[0,500,1000,1500,2000,3000]))['rate_per_mile'].mean())

distance
(0, 500]        2.564907
(500, 1000]     2.236443
(1000, 1500]    2.125952
(1500, 2000]    2.044124
(2000, 3000]    1.945001
Name: rate_per_mile, dtype: float64


C:\Users\pc gamer\AppData\Local\Temp\ipykernel_14892\1641985381.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(train_data.groupby(pd.cut(train_data['distance'],bins=[0,500,1000,1500,2000,3000]))['rate_per_mile'].mean())


In [17]:
train_data.groupby('equipment')['rate_per_mile'].mean()

equipment
Dry Van    2.115401
Flatbed    2.294729
Reefer     2.383376
Name: rate_per_mile, dtype: float64

In [18]:
train_data.groupby('pickup')['rate_per_mile'].mean().sort_values()

pickup
Reno              1.979448
San Francisco     2.011205
Fresno            2.046753
Albany            2.055319
Salt Lake City    2.056370
                    ...   
New Orleans       2.338994
Atlanta           2.340549
Nashville         2.352082
Chattanooga       2.378462
Dallas            2.425212
Name: rate_per_mile, Length: 64, dtype: float64

In [19]:
train_data.groupby('delivery')['rate_per_mile'].mean().sort_values()

delivery
Reno             1.991565
San Francisco    2.003797
Los Angeles      2.025262
Albany           2.063416
Bakersfield      2.074962
                   ...   
Little Rock      2.345919
Memphis          2.352426
Atlanta          2.353768
Nashville        2.357729
Birmingham       2.364832
Name: rate_per_mile, Length: 64, dtype: float64

In [20]:
train_data.groupby('pickup')[['pickup_lat','pickup_lon']].nunique().max()

pickup_lat    1
pickup_lon    1
dtype: int64

In [21]:
val_data.isnull().sum()


load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64

In [22]:
train_data.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate,dayofweek,rate_per_mile
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41,2,2.352935
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97,2,2.424135
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54,2,1.862513
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28,2,1.892770
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28,2,2.547112


In [23]:
val_data['date'] = pd.to_datetime(val_data['date'])
val_data['dayofweek'] = val_data['date'].dt.dayofweek

In [24]:
weight_median = train_data['weight'].median()
train_data['weight']= train_data['weight'].fillna(train_data['weight'].median())

val_data['weight'] = val_data['weight'].fillna(weight_median)

In [25]:
train_data['market_index'] = train_data['market_index'].fillna(train_data.groupby('dayofweek')['market_index'].transform('median'))

In [26]:
market_median = train_data.groupby('dayofweek')['market_index'].median()

val_data['market_index'] = val_data['market_index'].fillna(
    val_data['dayofweek'].map(market_median)
)

In [27]:
train_data = pd.get_dummies(train_data,columns=['equipment'])
val_data = pd.get_dummies(val_data,columns=['equipment'])


In [28]:
train_data = train_data.astype({col: int for col in train_data.select_dtypes(bool).columns})
val_data = val_data.astype({col: int for col in val_data.select_dtypes(bool).columns})

In [29]:
train_data = train_data.drop(columns=['load_id','pickup','delivery','date'])

In [30]:
val_id = val_data['load_id']
train_data = train_data.drop(columns=['rate_per_mile'])
val_data = val_data.drop(columns=['load_id', 'pickup', 'delivery','date'])

In [31]:
y = train_data['posted_rate']
X = train_data.drop(columns=['posted_rate'])
X_val = val_data

print(X.columns.tolist())
print(X_val.columns.tolist())

['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'market_index', 'quote_signal', 'dayofweek', 'equipment_Dry Van', 'equipment_Flatbed', 'equipment_Reefer']
['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'market_index', 'quote_signal', 'dayofweek', 'equipment_Dry Van', 'equipment_Flatbed', 'equipment_Reefer']


In [32]:
dates = pd.read_csv(r"C:\Users\pc gamer\Downloads\scotlab\train-test.csv")['date']
dates = pd.to_datetime(dates)

In [33]:
train_mask = dates < '2025-09-01'
X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[~train_mask]
y_test = y[~train_mask]
print(X_train.shape , X_test.shape)

(38477, 12) (9523, 12)


In [34]:
from sklearn.ensemble import RandomForestRegressor

In [35]:
rgs = RandomForestRegressor(random_state=42)
rgs.fit(X_train, y_train)
preds = rgs.predict(X_test)

In [36]:
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
r2 = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
print(r2)
print(mse)
print(mae)

0.8165935232220022
427123.5768013345
150.87733317231965


In [37]:
errors = pd.DataFrame({'actual': y_test, 'predicted': preds,'absolute error': (y_test - preds).abs(),'distance': X_test['distance'], 'dayofweek': X_test['dayofweek']})
errors.sort_values('absolute error', ascending=False).head(20)

,actual,predicted,absolute error,distance,dayofweek
40171,20361.89,4565.9483,15795.9417,2283.4,3
40096,20132.37,4401.1073,15731.2627,1760.3,3
46213,17893.17,4114.1620,13779.0080,2023.4,0
43539,17514.11,4047.0829,13467.0271,1893.0,4
47298,19110.30,6076.6777,13033.6223,2979.1,0
40322,14949.25,3129.2030,11820.0470,1639.9,4
46516,14784.38,4156.7425,10627.6375,1684.3,2
42149,13026.22,2790.8122,10235.4078,1415.5,2
44267,14425.42,4382.7278,10042.6922,2492.6,2
46366,14403.14,4386.5878,10016.5522,2251.9,1


In [38]:
train_data['rpm'] = train_data['posted_rate'] / train_data['distance']
print(train_data['rpm'].describe())
print((train_data['rpm'] > 5).sum(), "rows above $5/mile out of", len(train_data))

count    48000.000000
mean         2.215347
std          0.583887
min          0.333709
25%          1.986974
50%          2.145348
75%          2.342696
max         14.125294
Name: rpm, dtype: float64
317 rows above $5/mile out of 48000


In [39]:
outliers = train_data[train_data['rpm'] > 5]
print(outliers['equipment_Dry Van'].mean(), outliers['equipment_Flatbed'].mean(), outliers['equipment_Reefer'].mean())
print(train_data['equipment_Dry Van'].mean(), train_data['equipment_Flatbed'].mean(), train_data['equipment_Reefer'].mean())
print(outliers['dayofweek'].value_counts())
print(outliers['distance'].describe())

0.5646687697160884 0.1861198738170347 0.24921135646687698
0.5667083333333334 0.18235416666666668 0.2509375
dayofweek
2    62
3    52
1    51
6    46
0    42
5    33
4    31
Name: count, dtype: int64
count     317.000000
mean     1023.109148
std       668.635797
min        70.000000
25%       476.700000
50%       869.100000
75%      1421.300000
max      3127.200000
Name: distance, dtype: float64


In [40]:
print(train_data['rpm'].quantile([0.001, 0.005, 0.01, 0.99, 0.995, 0.999]))

0.001     0.437364
0.005     0.746195
0.010     1.673521
0.990     3.176940
0.995     6.314326
0.999    10.068775
Name: rpm, dtype: float64


In [41]:
low = train_data['rpm'].quantile(0.01)
high = train_data['rpm'].quantile(0.99)
train_clean = train_data[(train_data['rpm'] > low) & (train_data['rpm'] < high)]
print(len(train_data), "->", len(train_clean))

48000 -> 47040


In [42]:
train_clean = train_clean.drop(columns=['rpm'])
X = train_clean.drop(columns=['posted_rate'])
y = train_clean['posted_rate']

In [43]:
dates_clean = dates.loc[train_clean.index]
train_mask = dates_clean < '2025-09-01'

In [44]:
X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[~train_mask]
y_test = y[~train_mask]
print(X_train.shape , X_test.shape)

(37704, 12) (9336, 12)


In [45]:
X_full = train_data.drop(columns=['posted_rate', 'rpm'])
y_full = train_data['posted_rate']

test_mask_full = dates >= '2025-09-01'
X_test = X_full[test_mask_full]
y_test = y_full[test_mask_full]

print(X_test.shape)

(9523, 12)


In [46]:
print(X_train.shape, X_test.shape)

(37704, 12) (9523, 12)


In [47]:
rgs = RandomForestRegressor(random_state=42)
rgs.fit(X_train, y_train)
preds = rgs.predict(X_test)

In [48]:
r2 = r2_score(y_test, preds)
mse = mean_squared_error(y_test, preds)
mae = mean_absolute_error(y_test, preds)
print(r2)
print(mse)
print(mae)

0.8249995693186021
407547.2753608865
116.8833793447443


In [49]:
from xgboost import XGBRegressor
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

In [50]:
r2 = r2_score(y_test, xgb_preds)
mse = mean_squared_error(y_test, xgb_preds)
mae = mean_absolute_error(y_test, xgb_preds)
print(r2)
print(mse)
print(mae)

0.8228041794262014
412659.97802978236
140.8218143270938


In [51]:
xgb_mae = XGBRegressor(objective='reg:absoluteerror', random_state=42)
xgb_mae.fit(X_train, y_train)
xgb_mae_preds = xgb_mae.predict(X_test)

In [52]:
r2 = r2_score(y_test, xgb_mae_preds)
mse = mean_squared_error(y_test, xgb_mae_preds)
mae = mean_absolute_error(y_test, xgb_mae_preds)
print(r2)
print(mse)
print(mae)

0.8226879231738823
412930.71975705604
138.8635261375574


In [53]:
for leaf in [1, 2, 5, 10, 20]:
    rf = RandomForestRegressor(min_samples_leaf=leaf, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    p = rf.predict(X_test)
    print(leaf, round(r2_score(y_test, p), 4), round(mean_absolute_error(y_test, p), 2))

1 0.825 116.88
2 0.825 116.81
5 0.825 117.06
10 0.8248 118.3
20 0.8244 120.47


In [54]:
for n in [100, 300, 500]:
    rf = RandomForestRegressor(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    p = rf.predict(X_test)
    print(n, round(r2_score(y_test, p), 4), round(mean_absolute_error(y_test, p), 2))

100 0.825 116.88
300 0.825 116.84
500 0.8251 116.77


In [55]:
short = train_data[(train_data['distance'] > 300) & (train_data['distance'] < 420)]
print(short.groupby('dayofweek')['posted_rate'].mean())
print(len(short))

dayofweek
0    882.182761
1    888.722576
2    917.456131
3    905.495256
4    891.533371
5    903.927532
6    881.460942
Name: posted_rate, dtype: float64
3568


In [56]:
dec_features = [c for c in X_train.columns if c not in ['market_index', 'quote_signal']]

rf_dec = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf_dec.fit(X_train[dec_features], y_train)
dec_preds = rf_dec.predict(X_test[dec_features])

print(round(r2_score(y_test, dec_preds), 4))
print(round(mean_absolute_error(y_test, dec_preds), 2))

0.8282
106.92


In [69]:
dec = pd.read_csv(r"C:\Users\pc gamer\Downloads\scotlab\december-chart-inputs.csv")
dec.head()


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,NaN
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,NaN
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,NaN
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,NaN
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,NaN


In [68]:
df = pd.read_csv(r"C:\Users\pc gamer\Downloads\scotlab\train-test.csv")
lex= df[df['pickup']=='Lexington']
print(lex[['pickup_lat', 'pickup_lon']].head(1))

    pickup_lat  pickup_lon
28    36.99152   -84.99876


In [70]:
fort = df[df['pickup']=='Fort Wayne']
print(fort[['pickup_lat', 'pickup_lon']].head(1))

    pickup_lat  pickup_lon
55    41.31561   -85.36206


In [71]:
print(df[df['delivery']=='Fort Wayne'][['delivery_lat','delivery_lon']].head(1))

    delivery_lat  delivery_lon
24      41.31561     -85.36206


In [73]:
dec['date'] = pd.to_datetime(dec['date'])

new_dec = pd.DataFrame({
    'pickup_lat': 36.99152,
    'pickup_lon': -84.99876,
    'delivery_lat': 41.31561,
    'delivery_lon': -85.36206,
    'distance': 360.0,
    'weight': 32000.0,
    'dayofweek': dec['date'].dt.dayofweek,
    'equipment_Dry Van': 1,
    'equipment_Flatbed': 0,
    'equipment_Reefer': 0,
})
new_dec.head()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,dayofweek,equipment_Dry Van,equipment_Flatbed,equipment_Reefer
0,36.99152,-84.99876,41.31561,-85.36206,360.0,32000.0,0,1,0,0
1,36.99152,-84.99876,41.31561,-85.36206,360.0,32000.0,1,1,0,0
2,36.99152,-84.99876,41.31561,-85.36206,360.0,32000.0,2,1,0,0
3,36.99152,-84.99876,41.31561,-85.36206,360.0,32000.0,3,1,0,0
4,36.99152,-84.99876,41.31561,-85.36206,360.0,32000.0,4,1,0,0


In [74]:
print(new_dec.columns.tolist())
print(X_train[dec_features].columns.tolist())

['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'dayofweek', 'equipment_Dry Van', 'equipment_Flatbed', 'equipment_Reefer']
['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'weight', 'dayofweek', 'equipment_Dry Van', 'equipment_Flatbed', 'equipment_Reefer']


In [75]:
rf_dec


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [76]:
dec['predicted_rate'] = rf_dec.predict(new_dec)
print(dec[['date', 'predicted_rate']].head(10))

        date  predicted_rate
0 2025-12-01       827.29420
1 2025-12-02       832.84806
2 2025-12-03       833.93680
3 2025-12-04       832.51392
4 2025-12-05       827.61356
5 2025-12-06       826.28578
6 2025-12-07       821.46332
7 2025-12-08       827.29420
8 2025-12-09       832.84806
9 2025-12-10       833.93680


In [77]:
print(dec.columns.tolist())


['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']


In [79]:
dec['date'] = dec['date'].dt.strftime('%Y-%m-%d')
dec.to_csv('december-chart-inputs.csv', index=False)

AttributeError: Can only use .dt accessor with datetimelike values

In [80]:
print(dec['date'].dtype)
print(dec['date'].head(3))

object
0    2025-12-01
1    2025-12-02
2    2025-12-03
Name: date, dtype: object


In [81]:
dec.to_csv('december-chart-inputs.csv', index=False)

In [82]:
print(X_val.shape)
print(val_id.shape)

(12000, 12)
(12000,)


In [83]:
val_preds = rf_dec.predict(X_val[dec_features])

submission = pd.DataFrame({
    'load_id': val_id,
    'predicted_rate': val_preds
})

print(submission.shape)
print(submission.head())
print((submission['predicted_rate'] <= 0).sum())

(12000, 2)
     load_id  predicted_rate
0  TE-000001       836.05800
1  TE-000002      5183.02384
2  TE-000003      5166.56512
3  TE-000004      3987.59078
4  TE-000005      1859.95890
0


In [84]:
submission.to_csv('validation_predictions.csv', index=False)

In [85]:
submission.to_csv(r"C:\Users\pc gamer\Downloads\scotlab\validation_predictions.csv", index=False)
dec.to_csv(r"C:\Users\pc gamer\Downloads\scotlab\december-chart-inputs.csv", index=False)